# UCAP -- SAM 3 + OpenCV SGBM (stereo depth)

**End-to-end depth-gated anonymizer for a stereo egocentric pair** (left + right clips; `kam0` / `kam1` in the code below).

1. **SAM 3** segments the privacy concepts (`face`, `license plate`) on the **left** clip.
2. **Classical stereo matching** (OpenCV semi-global block matching) turns the pair into a disparity map -> the *near-object* mask (the interaction zone to **preserve**), with no learned depth model and no metric calibration.
3. The output redacts **privacy MINUS near**; significant carve-outs go to the **review log** + conflict video (**M2**) for the 3-way human decision.

> Runtime: Google Colab, a single T4 is enough.

## Step 1 -- runtime check

In [ ]:
# Runtime check -- confirm a GPU is attached (T4 is fine for stereo; the
# DAC/UniDepth notebook is happier on an L4 because two models are resident).
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("GPU:", name, "| total VRAM (GB):", round(total, 2))
    if not any(g in name for g in ("T4", "L4", "A100", "L40")):
        print("NOTE: unrecognised GPU -- numbers will not match the T4 headline.")
else:
    print("No GPU. Runtime > Change runtime type > GPU (T4 or L4).")

## Step 2 -- install

In [ ]:
# Installs. SAM 3 (ultralytics) + OpenCV (StereoSGBM is built in).
# opencv-contrib adds the WLS disparity filter (optional, nicer masks).
!pip install -q -U ultralytics huggingface_hub
!pip install -q -U opencv-contrib-python-headless imageio imageio-ffmpeg matplotlib
print("install done")

## Step 3 -- Hugging Face auth (gated SAM 3)

In [ ]:
# Hugging Face auth -- facebook/sam3 is a GATED repo.
#   1. Request access once: https://huggingface.co/facebook/sam3
#   2. Token (read scope): https://huggingface.co/settings/tokens
#   3. Colab: Secrets panel (key icon) -> add secret HF_TOKEN -> enable for notebook.
# NEVER hardcode a token in a cell -- it leaks the moment the file is shared.
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
    print("HF auth OK -- gated SAM 3 weights accessible.")
except Exception as e:
    print("HF auth NOT set up:", repr(e))
    print("SAM 3 (sam3.pt) will fail to download without it.")

## Step 4 -- config

In [ ]:
# ============================ CONFIG (edit me) =============================
# EgoBlur-matched released pair; widen by adding noun-phrase prompts if needed.
PROMPTS = ["face", "license plate"]
# PROMPTS += ["document", "computer screen", "phone screen", "tattoo"]

SCORE_THRESHOLD = 0.6        # recall-first (paper configuration)
MAX_FRAMES      = None       # cap for a quick pass; None = whole clip

# --- Near zone from STEREO disparity (the zone to PRESERVE) ---------------
# IMPORTANT (your calibration question): a near-object MASK needs NO calibration.
# Default = relative -- the nearest NEAR_FRACTION_PCT% of matched pixels (largest
# disparity = closest). You only need focal+baseline to express it in real metres.
USE_METRIC       = False     # False: relative (no calibration). True: metres via focal+baseline.
NEAR_FRACTION_PCT = 40       # relative: nearest 40% of valid-disparity pixels are "near"
NEAR_METERS      = 1.0       # used only if USE_METRIC=True
FOCAL_PX         = None      # rectified focal length in pixels (fill in for metric)
BASELINE_M       = 0.06      # ~6 cm kam0<->kam1 baseline (confirm the exact value)

# --- Carve policy + review threshold (this is M2) -------------------------
PRESERVE_POLICY  = "carve"   # "carve" = blur MINUS near | "failsafe" = blur all, still flag
OVERLAP_THRESHOLD = 0.15

# --- Redaction style ------------------------------------------------------
REDACTION       = "blur"     # "blur" | "pixelate" | "fill"
BLUR_KSIZE      = 41
PIXELATE_BLOCKS = 16
FILL_BGR        = (255, 0, 0)

# --- Outputs --------------------------------------------------------------
OUT_VIDEO       = "/content/anon_stereo.mp4"
REVIEW_LOG      = "/content/review_log_stereo.json"
SAMPLE_FIG_PATH = "/content/anon_stereo_sample.jpg"
print("config set | policy:", PRESERVE_POLICY, "| near:",
      (str(NEAR_METERS) + " m") if USE_METRIC else ("nearest " + str(NEAR_FRACTION_PCT) + "%"))

## Step 5 -- stereo pair (left + right)

In [ ]:
# Stereo pair -- LEFT drives SAM 3; RIGHT is the depth partner. kam0/kam1 are
# the ~6 cm CogniCap pair (upload clips/cognicap/*_short_compressed.mp4 first).
import cv2, numpy as np

LEFT_PATH = RIGHT_PATH = None
# Option 1 -- Drive:
# from google.colab import drive; drive.mount("/content/drive")
# LEFT_PATH  = "/content/drive/MyDrive/kam0_short_compressed.mp4"
# RIGHT_PATH = "/content/drive/MyDrive/kam1_short_compressed.mp4"
# Option 2 -- upload both:
# from google.colab import files; up = files.upload()
# LEFT_PATH, RIGHT_PATH = "kam0_short_compressed.mp4", "kam1_short_compressed.mp4"
# Option 3 -- repo clips (upload them next to this notebook):
# LEFT_PATH, RIGHT_PATH = "kam0_short_compressed.mp4", "kam1_short_compressed.mp4"

# Option 4 -- synthetic stereo pair (a box with a small L/R disparity):
if LEFT_PATH is None:
    def _synth(path, shift):
        vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 30, (640, 480))
        for i in range(60):
            a = np.full((480, 640, 3), 230, np.uint8)
            x = 20 + i * 8
            cv2.rectangle(a, (x + shift, 200), (x + shift + 110, 320), (40, 90, 200), -1)
            vw.write(a)
        vw.release()
    LEFT_PATH, RIGHT_PATH = "synth_left.mp4", "synth_right.mp4"
    _synth(LEFT_PATH, 0); _synth(RIGHT_PATH, -12)   # box ~12 px nearer-looking
    print("Synthetic stereo pair -- use Option 1/2/3 for real numbers.")

VIDEO_PATH = LEFT_PATH      # the pipeline streams SAM 3 over the LEFT clip
print("LEFT:", LEFT_PATH, "| RIGHT:", RIGHT_PATH)

## Step 6 -- helpers

In [ ]:
# Shared helpers: VRAM, redaction, and the figure utility.
import time, gc, json
import numpy as np
import cv2
import matplotlib.pyplot as plt

def reset_vram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

def peak_vram_gb():
    return round(torch.cuda.max_memory_allocated() / 1e9, 3) if torch.cuda.is_available() else None

def apply_redaction(frame_bgr, mask):
    """Redact the True pixels of `mask` using the REDACTION mode from config.
    Mask-aware: only masked pixels change, so background detail is untouched."""
    out = frame_bgr.copy()
    if mask is None or not mask.any():
        return out
    if REDACTION == "fill":
        out[mask] = FILL_BGR
    elif REDACTION == "pixelate":
        h, w = frame_bgr.shape[:2]
        s = max(1, PIXELATE_BLOCKS)
        small = cv2.resize(frame_bgr, (max(1, w // s), max(1, h // s)),
                           interpolation=cv2.INTER_LINEAR)
        pix = cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
        out[mask] = pix[mask]
    else:  # "blur" (default) -- matches code/pipeline.py blur_regions
        k = int(BLUR_KSIZE) | 1
        out[mask] = cv2.GaussianBlur(frame_bgr, (k, k), 0)[mask]
    return out

def depth_to_color(depth):
    """Normalised inferno heatmap of a depth/disparity map for figures."""
    d = np.asarray(depth, dtype="float32")
    finite = np.isfinite(d) & (d > 0)
    if not finite.any():
        return np.zeros((*d.shape, 3), "uint8")
    lo, hi = np.percentile(d[finite], [2, 98])
    n = np.clip((d - lo) / max(hi - lo, 1e-6), 0, 1)
    return (plt.cm.inferno(n)[:, :, :3] * 255).astype("uint8")

SAMPLES = []    # (idx, redacted_bgr, depth, near_mask, privacy_mask) for the figure
CONFLICTS = []  # (idx, time_s, redacted_bgr, deblurred_mask) for the conflict reel (Step 10b)
print("helpers ready")

## Step 7 -- depth backend (stereo SGBM)
Disparity from the synced pair; the right clip is read in lockstep with the SAM 3 stream.

In [ ]:
# === Depth backend: stereo matching (OpenCV SGBM) on kam0 (L) vs kam1 (R) ===
# Disparity per frame from the synced pair. The RIGHT clip is read in lockstep
# with the SAM 3 stream over the LEFT clip (frame i <-> frame i).
import cv2, numpy as np

# SGBM params -- tune num_disp (multiple of 16) to the rig's max disparity.
_min_disp, _num_disp, _block = 0, 16 * 6, 5
_sgbm = cv2.StereoSGBM_create(
    minDisparity=_min_disp, numDisparities=_num_disp, blockSize=_block,
    P1=8 * 3 * _block ** 2, P2=32 * 3 * _block ** 2, disp12MaxDiff=1,
    uniquenessRatio=10, speckleWindowSize=100, speckleRange=2,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)

_right_cap = cv2.VideoCapture(RIGHT_PATH)
_last_right = [None]

def estimate_depth(frame_left_bgr, idx):
    """Return a disparity map (H, W) for the LEFT frame; larger = closer.
    NOTE: raw (unrectified) SGBM. For a relative near-mask that is enough; for
    accurate metric depth, rectify the pair first (needs calibration)."""
    if idx == 0:
        _right_cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    ok, right = _right_cap.read()
    if not ok:
        right = _last_right[0]
    if right is None:
        return np.zeros(frame_left_bgr.shape[:2], "float32")
    _last_right[0] = right
    gl = cv2.cvtColor(frame_left_bgr, cv2.COLOR_BGR2GRAY)
    gr = cv2.cvtColor(right, cv2.COLOR_BGR2GRAY)
    if gr.shape != gl.shape:
        gr = cv2.resize(gr, (gl.shape[1], gl.shape[0]))
    disp = _sgbm.compute(gl, gr).astype("float32") / 16.0   # SGBM scales by 16
    disp[disp < 0] = 0
    return disp

def near_mask(disp):
    """Bool mask of the near interaction zone to PRESERVE."""
    d = np.asarray(disp, dtype="float32")
    if USE_METRIC and FOCAL_PX and BASELINE_M:
        with np.errstate(divide="ignore", invalid="ignore"):
            Z = (FOCAL_PX * BASELINE_M) / np.where(d > 0, d, np.nan)
        return np.isfinite(Z) & (Z < NEAR_METERS)
    valid = d[d > 0]
    if valid.size == 0:
        return np.zeros(d.shape, bool)
    return d >= np.percentile(valid, 100 - NEAR_FRACTION_PCT)   # largest disp = nearest

# Smoke test on frame 0:
_capL = cv2.VideoCapture(LEFT_PATH); _ok, _fL = _capL.read(); _capL.release()
if _ok:
    _disp = estimate_depth(_fL, 0)
    print("disparity:", _disp.shape, "| max %.1f" % float(_disp.max()),
          "| near pixels:", int(near_mask(_disp).sum()))

## Step 8 -- load SAM 3

In [ ]:
# Load SAM 3 once -- Ultralytics video-native predictor (axis B1, the
# BrainHack-validated path). `sam3.pt` is the gated facebook/sam3 weight.
from huggingface_hub import hf_hub_download
from ultralytics.models.sam import SAM3VideoSemanticPredictor

SAM3_PT = hf_hub_download(repo_id="facebook/sam3", filename="sam3.pt", local_dir=".")
print("sam3.pt:", SAM3_PT)

# half=True -> FP16 (the T4 path); retina_masks=True -> full-res instance masks
# aligned to orig_img; score_threshold_detection is the recall lever (0.25-0.6).
sam3_overrides = dict(task="segment", mode="predict", model="sam3.pt",
                      half=True, save=False, retina_masks=True, verbose=False)
sam3 = SAM3VideoSemanticPredictor(overrides=sam3_overrides,
                                  score_threshold_detection=SCORE_THRESHOLD)
print("SAM 3 ready. Prompts:", PROMPTS)

## Step 9 -- run the pipeline

In [ ]:
# === The pipeline: SAM 3 masks -> depth near-mask -> (blur MINUS near) ======
# One streaming SAM 3 pass over the LEFT/only video. Per frame:
#   privacy = union of SAM 3 instance masks for PROMPTS
#   near    = near_mask(estimate_depth(frame, idx))    # the zone to preserve
#   blur    = privacy & ~near   (PRESERVE_POLICY="carve", the requested behaviour)
#          or privacy           (PRESERVE_POLICY="failsafe": redact all, still flag)
# A frame is flagged for review when ANY privacy instance has >= OVERLAP_THRESHOLD
# of its area inside the near zone -- exactly resolve_conflicts() in pipeline.py.
def _instances(r, H, W):
    out = []
    m = getattr(r, "masks", None)
    if m is not None and getattr(m, "data", None) is not None and len(m.data) > 0:
        arr = m.data.to("cpu").numpy().astype(bool)            # (N, h, w)
        for k in range(arr.shape[0]):
            im = arr[k]
            if im.shape != (H, W):
                im = cv2.resize(im.astype("uint8"), (W, H),
                                interpolation=cv2.INTER_NEAREST).astype(bool)
            out.append(im)
    return out

def process_video(video_path, out_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))

    review = []
    SAMPLES.clear()
    CONFLICTS.clear()
    n = n_blur = n_flag = 0
    sample_at = set(np.linspace(0, max(0, (MAX_FRAMES or 120) - 1), 3, dtype=int).tolist())
    reset_vram(); t0 = time.perf_counter()

    for r in sam3(source=video_path, text=PROMPTS, stream=True):
        frame = r.orig_img.copy()                              # BGR, full size
        Hf, Wf = frame.shape[:2]
        insts = _instances(r, Hf, Wf)
        privacy = np.zeros((Hf, Wf), bool)
        for im in insts:
            privacy |= im

        try:
            depth = estimate_depth(frame, n)
            near = near_mask(depth)
            if near.shape != (Hf, Wf):
                near = cv2.resize(near.astype("uint8"), (Wf, Hf),
                                  interpolation=cv2.INTER_NEAREST).astype(bool)
        except Exception as e:
            if n == 0:
                print("depth error (continuing with empty near-mask):", repr(e))
            depth, near = np.zeros((Hf, Wf), "float32"), np.zeros((Hf, Wf), bool)

        blur = (privacy & ~near) if PRESERVE_POLICY == "carve" else privacy.copy()

        flagged, max_ov = False, 0.0
        for im in insts:
            d = int(im.sum())
            if not d:
                continue
            ov = float((im & near).sum()) / d
            max_ov = max(max_ov, ov)
            if ov >= OVERLAP_THRESHOLD:
                flagged = True
        if flagged:
            n_flag += 1
            review.append({"frame": n, "time_s": round(n / fps, 3),
                           "max_overlap": round(max_ov, 3),
                           "deblurred_px": int((privacy & near).sum()),
                           "privacy_px": int(privacy.sum())})

        out = apply_redaction(frame, blur)
        if blur.any():
            n_blur += 1
        writer.write(out)
        if n in sample_at:
            SAMPLES.append((n, out.copy(), depth, near, privacy))
        if flagged:
            CONFLICTS.append((n, round(n / fps, 3), out.copy(), (privacy & near)))
        n += 1
        if MAX_FRAMES and n >= MAX_FRAMES:
            break
    writer.release()
    dt = time.perf_counter() - t0
    stats = {"frames": n, "fps_pipeline": round(n / dt, 2) if dt else 0.0,
             "frames_with_blur": n_blur, "flagged_frames": n_flag,
             "m2_conflict_rate": round(n_flag / n, 4) if n else 0.0,
             "peak_vram_gb": peak_vram_gb(), "out": out_path}
    return review, stats

REVIEW, STATS = process_video(VIDEO_PATH, OUT_VIDEO)
print(json.dumps(STATS, indent=2, default=str))

## Step 10 -- review log + M2

In [ ]:
# === Review log + the M2 conflict/review number =============================
# REVIEW is the human reviewer's worklist: the timestamps where the near-object
# carve-out was significant (a privacy surface met the interaction zone). This is
# M2 -- conflict / human-review rate -- made concrete (code/metrics.py).
with open(REVIEW_LOG, "w", encoding="utf-8") as fh:
    json.dump({"video": VIDEO_PATH, "prompts": PROMPTS,
               "preserve_policy": PRESERVE_POLICY,
               "overlap_threshold": OVERLAP_THRESHOLD,
               "stats": STATS, "events": REVIEW}, fh, indent=2, default=str)

print("M2 conflict/review rate:", STATS["m2_conflict_rate"],
      "(" + str(STATS["flagged_frames"]) + "/" + str(STATS["frames"]) + " frames flagged)")
print("review log ->", REVIEW_LOG)
print()
print("first review events (frame @ time_s : max_overlap):")
for e in REVIEW[:12]:
    print(f"  frame {e['frame']:4d} @ {e['time_s']:7.2f}s : "
          f"overlap {e['max_overlap']:.2f}, deblurred {e['deblurred_px']} px")
if not REVIEW:
    print("  (none -- no privacy surface entered the near zone at this threshold)")

## Step 10b -- conflict review reel
A video of **only the conflict frames** (the ones in the review log), each with a
**red border around the de-blurred areas** -- the sensitive regions kept sharp
because they sit in the near interaction zone. This is the reviewer's visual
worklist. It reads `CONFLICTS`, which **Step 9 now stashes**, so **re-run Step 9
once** after adding this before running this cell.

In [ ]:
# === Conflict review reel: red border around every de-blurred area =========
# A video of ONLY the conflict frames (those in the review log), each showing the
# REDACTED output with a RED outline around the de-blurred regions -- the privacy
# pixels kept sharp because they fell in the near / interaction zone. Every red
# outline is something a human should eyeball. Fed by CONFLICTS, which Step 9 now
# stashes -> re-run Step 9 first if you have not since adding this cell.
REEL_VIDEO = OUT_VIDEO.replace(".mp4", "_conflicts.mp4")
REEL_FPS   = 4              # low fps so each conflict frame lingers (~1/REEL_FPS s)
BORDER_BGR = (0, 0, 255)    # red (OpenCV BGR)
BORDER_PX  = 3

if not CONFLICTS:
    print("No conflict frames to render -- review log was empty. "
          "(Run Step 9 first, or no privacy surface met the near zone.)")
else:
    H, W = CONFLICTS[0][2].shape[:2]
    rw = cv2.VideoWriter(REEL_VIDEO, cv2.VideoWriter_fourcc(*"mp4v"), REEL_FPS, (W, H))
    for idx, t_s, red, deblurred in CONFLICTS:
        canvas = red.copy()
        m = np.asarray(deblurred).astype("uint8")
        if m.shape[:2] != canvas.shape[:2]:
            m = cv2.resize(m, (canvas.shape[1], canvas.shape[0]),
                           interpolation=cv2.INTER_NEAREST)
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(canvas, cnts, -1, BORDER_BGR, BORDER_PX)
        label = "frame %d  t=%.2fs  de-blurred regions: %d" % (idx, t_s, len(cnts))
        cv2.rectangle(canvas, (0, 0), (W, 26), (0, 0, 0), -1)
        cv2.putText(canvas, label, (6, 18), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    (255, 255, 255), 1, cv2.LINE_AA)
        rw.write(canvas)
    rw.release()
    print("conflict reel ->", REEL_VIDEO, "| frames:", len(CONFLICTS), "| fps:", REEL_FPS)
    print("Download:  from google.colab import files; files.download('" + REEL_VIDEO + "')")

## Step 11 -- sample figure (redacted only)

In [ ]:
# === Sample figure -- REDACTED output only (no raw PII; context/README.md) ==
# Columns: redacted frame | depth/disparity heatmap | near-mask (cyan) over the
# redacted frame. Safe to put in the paper / show the supervisor.
if SAMPLES:
    fig, axes = plt.subplots(len(SAMPLES), 3, figsize=(13, 4 * len(SAMPLES)))
    if len(SAMPLES) == 1:
        axes = axes[None, :]
    for row, (idx, red, depth, near, priv) in enumerate(SAMPLES):
        rgb = cv2.cvtColor(red, cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(rgb); axes[row, 0].set_title("frame %d -- redacted" % idx)
        axes[row, 1].imshow(depth_to_color(depth)); axes[row, 1].set_title("depth / disparity")
        ov = rgb.copy()
        if near is not None and near.any():
            ov[near] = (0.5 * ov[near] + 0.5 * np.array([0, 255, 255])).astype("uint8")
        axes[row, 2].imshow(ov); axes[row, 2].set_title("near zone (preserved) = cyan")
        for c in range(3):
            axes[row, c].axis("off")
    plt.tight_layout(); plt.savefig(SAMPLE_FIG_PATH, dpi=110, bbox_inches="tight")
    plt.show()
    print("figure ->", SAMPLE_FIG_PATH)
else:
    print("no samples captured")

## Step 12 -- save / download

In [ ]:
# === Save / download the three outputs ======================================
# Recommended: copy to Drive so they survive the runtime. Otherwise download.
print("outputs:")
print("  video :", OUT_VIDEO)
print("  log   :", REVIEW_LOG)
print("  figure:", SAMPLE_FIG_PATH)

# --- Option A: copy to Google Drive (uncomment) ---
# from google.colab import drive; drive.mount("/content/drive")
# import shutil, os
# dst = "/content/drive/MyDrive/anon_out"; os.makedirs(dst, exist_ok=True)
# for p in (OUT_VIDEO, REVIEW_LOG, SAMPLE_FIG_PATH): shutil.copy(p, dst)
# print("copied to", dst)

# --- Option B: download to your machine (uncomment) ---
# from google.colab import files
# files.download(OUT_VIDEO); files.download(REVIEW_LOG); files.download(SAMPLE_FIG_PATH)

---
### After it runs
1. Watch `OUT_VIDEO`: privacy surfaces blurred, the near interaction zone left sharp.
2. Open `REVIEW_LOG`: each event is a frame a human should check -- this is **M2** (conflict / human-review). The printed `m2_conflict_rate` is the headline number; pair it with M1/M3 (leak) for the trade-off.
3. Tune from the **config cell**: `PROMPTS`, the near threshold, `OVERLAP_THRESHOLD`, and `PRESERVE_POLICY` ("carve" vs "failsafe").

**The privacy nuance (your thesis spine).** `"carve"` keeps a near, held object sharp -- but if that object IS a privacy surface (a phone screen showing a face in your hand), carving it un-blurs PII -> an M1/M3 leak. `"failsafe"` blurs all PII and only *flags* the overlap. The review log is what lets you defend either choice with a number.